# Import Libraries

In [1]:
import pandas as pd
import torch
import pickle
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from tqdm import tqdm

c:\Users\VICTUS\OneDrive\Documents\Semester_4\ML\ML\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tqdm.pandas() # Progress Bar for Pandas

# Load Model

As discussed in `08_validation_performance.ipynb`, the baseline DistilBERT model achieved the best balance between computational efficiency and overall performance. Therefore, we use the baseline model for the first implementation, which is self-labeling movie synopses so that each movie obtains a new emotion classification column.

In [3]:
path_model = '../models/bert' 
tokenizer = DistilBertTokenizer.from_pretrained(path_model)
model = DistilBertForSequenceClassification.from_pretrained(path_model)

path_encoder = '../models/bert/label_encoder.pkl'
with open(path_encoder, 'rb') as f:
    label_encoder = pickle.load(f)

# Load Dataset IMDB

In [4]:
df_movies = pd.read_csv('../data/imdb_train.csv')
df_movies.head(5)

,Series_Title,Genre,Overview,clean_overview
0,The Shawshank Redemption,Drama,Two imprisoned men bond over a number of years...,two imprisoned men bond over a number of years...
1,The Godfather,"Crime, Drama",An organized crime dynasty's aging patriarch t...,an organized crime dynasty's aging patriarch t...
2,The Dark Knight,"Action, Crime, Drama",When the menace known as the Joker wreaks havo...,when the menace known as the joker wreaks havo...
3,The Godfather: Part II,"Crime, Drama",The early life and career of Vito Corleone in ...,the early life and career of vito corleone in ...
4,12 Angry Men,"Crime, Drama",A jury holdout attempts to prevent a miscarria...,a jury holdout attempts to prevent a miscarria...


# Predict Emotion based on Overview

In [5]:
def predict_emotion(text):
    # Limit text length to 128 tokens
    inputs = tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    
    # Disable gradient calculation for faster inference
    with torch.no_grad():
        outputs = model(**inputs)
        
    prediction_idx = torch.argmax(outputs.logits, dim=1).item()
    
    # Convert prediction index back to emotion label
    return label_encoder.inverse_transform([prediction_idx])[0]

In [6]:
# Predicting in 'clean_overview'
df_movies['predicted_emotion'] = df_movies['clean_overview'].progress_apply(predict_emotion)

100%|██████████| 1000/1000 [00:19<00:00, 50.08it/s]


# Result

In [7]:
df_movies[['Series_Title', 'Overview', 'predicted_emotion']].head()

,Series_Title,Overview,predicted_emotion
0,The Shawshank Redemption,Two imprisoned men bond over a number of years...,Sadness
1,The Godfather,An organized crime dynasty's aging patriarch t...,Anger
2,The Dark Knight,When the menace known as the Joker wreaks havo...,Fear
3,The Godfather: Part II,The early life and career of Vito Corleone in ...,Joy
4,12 Angry Men,A jury holdout attempts to prevent a miscarria...,Sadness


In [8]:
df_movies['predicted_emotion'].value_counts()

predicted_emotion
Fear        274
Sadness     239
Anger       220
Joy         112
Love         80
Surprise     75
Name: count, dtype: int64

# Saving the Updated Dataset

In [9]:
df_movies.to_csv('../data/imdb_movies_with_emotions.csv', index=False)